In [3]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import mlflow
import mlflow.pytorch
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
import os

# Create the parent target directory if it does not exist yet
os.makedirs("./models", exist_ok=True)

# ==========================================================
# 1. MLFLOW CONFIGURATION INTERFACE
# ==========================================================
mlflow.set_tracking_uri('http://127.0.0.1:5000')
mlflow.set_experiment('Deep Learning')

# Hardware acceleration allocation 
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
np.random.seed(42)
torch.manual_seed(42)

# ==========================================================
# 2. INGEST DATASET & IN-MEMORY SIGNAL INJECTION
# ==========================================================
dfc = pd.read_csv(r"D:\Customer Support\Data set\customer_support_tickets_FE.csv")
dfc.columns = dfc.columns.str.strip()

# Zero-indexed structural mapping transformations [1-5] -> [0-4]
dfc['Ticket Type'] = dfc['Ticket Type'].astype(int) - 1
dfc['Ticket Description'] = dfc['Ticket Description'].fillna("missing description").astype(str).str.lower()

# -------------------------------------------------------------------------
# Dynamic vocabulary construction mapping index loops
# -------------------------------------------------------------------------
words = []
for desc in dfc['Ticket Description']:
    words.extend(desc.split())

word_counts = Counter(words)
# Keep top 10k vocabulary keywords
vocab = {word: i + 2 for i, (word, count) in enumerate(word_counts.most_common(10000))}
vocab["<PAD>"] = 0
vocab["<UNK>"] = 1

def text_to_sequence(text, max_len=100):
    tokens = text.split()
    seq = [vocab.get(token, 1) for token in tokens[:max_len]]
    return seq + [0] * (max_len - len(seq))

# Transform textual values into structured dense arrays
sequences = np.array([text_to_sequence(txt) for txt in dfc['Ticket Description']])

# ==========================================================
# 3. EXTRACTION AND STRATIFIED VALIDATION SPLITTING
# ==========================================================
X_train, X_val, y_train, y_val = train_test_split(
    sequences, dfc['Ticket Type'].values, test_size=0.2, stratify=dfc['Ticket Type'].values, random_state=42
)

class SequenceDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.long)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_loader = DataLoader(SequenceDataset(X_train, y_train), batch_size=32, shuffle=True)
val_loader = DataLoader(SequenceDataset(X_val, y_val), batch_size=64, shuffle=False)

# ==========================================================
# 4. PYTORCH BiLSTM GRAPH NETWORK LAYOUT STRUCTURE
# ==========================================================
class BiLSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes):
        super(BiLSTMClassifier, self).__init__()
        # 100d embedding layer representing the GloVe dimension constraint
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True, bidirectional=True, num_layers=1)
        
        # Regularization Components
        self.bn = nn.BatchNorm1d(hidden_dim * 2)
        self.dropout = nn.Dropout(0.4)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)
        
    def forward(self, x):
        embedded = self.embedding(x)
        lstm_out, (hidden, cell) = self.lstm(embedded)
        
        # Max pool over spatial sequence length dimensions
        pooled, _ = torch.max(lstm_out, dim=1)
        
        normed = self.bn(pooled)
        dropped = self.dropout(normed)
        return self.fc(dropped)

# Instantiate network parameters
model = BiLSTMClassifier(vocab_size=len(vocab), embed_dim=100, hidden_dim=64, num_classes=5).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# ==========================================================
# 5. EXECUTION COMPILER LOOP WITH EARLY STOPPING REGISTRATION
# ==========================================================
print("Launching Deep Learning Training Process (BiLSTM Task)...")
epochs = 15
best_val_loss = float('inf')
patience, patience_counter = 3, 0

with mlflow.start_run(run_name="BiLSTM_Sequence_TicketType"):
    mlflow.log_param("embedding_dimensions", 100)
    mlflow.log_param("hidden_units", 64)
    mlflow.log_param("dropout_rate", 0.4)
    
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        
        for batch_x, batch_y in train_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            optimizer.zero_grad()
            
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            
        # Validation Loop Validation Metric Checkpoints
        model.eval()
        val_loss = 0
        all_preds, all_labels, all_probs = [], [], []
        
        with torch.no_grad():
            for batch_x, batch_y in val_loader:
                batch_x, batch_y = batch_x.to(device), batch_y.to(device)
                outputs = model(batch_x)
                loss = criterion(outputs, batch_y)
                val_loss += loss.item()
                
                probs = torch.softmax(outputs, dim=1).cpu().numpy()
                preds = torch.argmax(outputs, dim=1).cpu().numpy()
                
                all_preds.extend(preds)
                all_labels.extend(batch_y.cpu().numpy())
                all_probs.extend(probs)
        
        avg_train_loss = total_loss / len(train_loader)
        avg_val_loss = val_loss / len(val_loader)
        accuracy = accuracy_score(all_labels, all_preds)
        f1_macro = f1_score(all_labels, all_preds, average='macro')
        
        print(f"Epoch {epoch+1:02d} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Accuracy: {accuracy:.4f}")
        
        mlflow.log_metric("training_loss", avg_train_loss, step=epoch)
        mlflow.log_metric("validation_loss", avg_val_loss, step=epoch)
        mlflow.log_metric("epoch_accuracy", accuracy, step=epoch)
        
        # Early Stopping check conditions
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            patience_counter = 0
            # Track best parameter checkpoints
            torch.save(model.state_dict(), "./models/best_bilstm_state.pt")
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"[EARLY STOPPING ENFORCED] Training loop halted at epoch {epoch+1}")
                break

    # Final out-of-sample metrics evaluations logged directly to artifacts
    final_roc_auc = roc_auc_score(all_labels, np.array(all_probs), multi_class='ovr')
    mlflow.log_metric("final_validation_accuracy", accuracy)
    mlflow.log_metric("final_macro_f1_score", f1_macro)
    mlflow.log_metric("final_roc_auc_ovr", final_roc_auc)
    
    mlflow.pytorch.log_model(model, artifact_path="bilstm_sequential_model")
    print("[SUCCESS] Deep learning pipeline fully tracked. Run 'mlflow ui' to audit metrics graphs.")

Launching Deep Learning Training Process (BiLSTM Task)...


2026/06/26 06:57:47 WARNING mlflow.utils.git_utils: Failed to import Git (the Git executable is probably not on your PATH), so Git SHA is not available. Error: Failed to initialize: Bad git executable.
The git executable must be specified in one of the following ways:
    - be included in your $PATH
    - be set via $GIT_PYTHON_GIT_EXECUTABLE
    - explicitly set via git.refresh(<full-path-to-git-executable>)

All git commands will error until this is rectified.

This initial message can be silenced or aggravated in the future by setting the
$GIT_PYTHON_REFRESH environment variable. Use one of the following values:
    - quiet|q|silence|s|silent|none|n|0: for no message or exception
    - warn|w|warning|log|l|1: for a warning message (logging level CRITICAL, displayed by default)
    - error|e|exception|raise|r|2: for a raised exception

Example:
    export GIT_PYTHON_REFRESH=quiet



Epoch 01 | Train Loss: 1.7304 | Val Loss: 1.6313 | Accuracy: 0.2019
Epoch 02 | Train Loss: 1.6147 | Val Loss: 1.6257 | Accuracy: 0.2090
Epoch 03 | Train Loss: 1.5845 | Val Loss: 1.6387 | Accuracy: 0.1824
Epoch 04 | Train Loss: 1.5577 | Val Loss: 1.6532 | Accuracy: 0.2048
Epoch 05 | Train Loss: 1.5296 | Val Loss: 1.6658 | Accuracy: 0.2054
[EARLY STOPPING ENFORCED] Training loop halted at epoch 5


2026/06/26 07:01:17 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/26 07:01:18 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.


[SUCCESS] Deep learning pipeline fully tracked. Run 'mlflow ui' to audit metrics graphs.
🏃 View run BiLSTM_Sequence_TicketType at: http://127.0.0.1:5000/#/experiments/5/runs/25fa123b339f4a99abd61a3f018dcb0f
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/5
